# Lesson 4: Organising Code

**Week 3 · Data Engineering Course**

---

As your pipeline grows, keeping all the code in one notebook becomes messy. This lesson covers three tools for organising code:

1. **The standard library** — useful modules that come with Python
2. **Classes** — a way to bundle related data and functions together
3. **Modules** — separate `.py` files for reusable code

**What you will build:** a small, well-structured cleaning pipeline that you could hand to another engineer and they could understand and run without your help.

In [ ]:
import csv
import json
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

RAW = Path('../data/raw')
print('Ready.')

---

## 4.1 The Standard Library

Python ships with a large library of useful modules. You do not need to install them — just `import` them.

In [ ]:
# datetime: work with dates
from datetime import datetime

# Parse a date string
date_str = '2023-07-14'
parsed = datetime.strptime(date_str, '%Y-%m-%d')
print(f'Parsed: {parsed}')
print(f'Year: {parsed.year}, Month: {parsed.month}, Day: {parsed.day}')

# Format a date as a string
formatted = parsed.strftime('%Y-%m-%d')
print(f'Formatted back: {formatted}')

In [ ]:
# More datetime operations you will use regularly
parsed = datetime.strptime('2023-07-14', '%Y-%m-%d')

# Month as a name (July, not 7)
month_name = parsed.strftime('%B')
print(f'Month name: {month_name}')       # July

# Short month name
short_month = parsed.strftime('%b')
print(f'Short month: {short_month}')     # Jul

# Day of the week (Monday=0, Sunday=6)
print(f'Weekday number: {parsed.weekday()}')          # 4 = Friday
print(f'Weekday name: {parsed.strftime("%A")}')       # Friday

# Comparing two dates
date_a = datetime.strptime('2023-01-05', '%Y-%m-%d')
date_b = datetime.strptime('2023-07-14', '%Y-%m-%d')
print(f'\ndate_a < date_b: {date_a < date_b}')   # True — January is before July
print(f'Days between: {(date_b - date_a).days}')  # 190

In [ ]:
# Handle multiple date formats — common in messy data
def parse_date(date_str):
    '''Try several date formats. Return YYYY-MM-DD string, or None if none match.'''
    formats = ['%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y', '%b %d %Y']
    for fmt in formats:
        try:
            return datetime.strptime(str(date_str).strip(), fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue   # try the next format
    return None   # nothing matched

test_dates = ['2023-01-05', '2023/07/14', '15/01/2023', 'Jan 20 2023', 'bad-input']
for d in test_dates:
    result = parse_date(d)
    print(f'  {d!r:22} -> {result}')

In [ ]:
# Counter: count occurrences in one line
from collections import Counter

with open(RAW / 'sales_q1.csv', newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

category_counts = Counter(row['category'].strip() for row in rows)
print('Category distribution (raw):')
for cat, count in category_counts.most_common():
    print(f'  {cat!r}: {count}')

In [ ]:
# defaultdict: a dict that creates a default value for missing keys
# Useful for grouping rows without checking if a key exists first
from collections import defaultdict

rows_by_category = defaultdict(list)
for row in rows:
    cat = row['category'].strip().title()
    rows_by_category[cat].append(row['order_id'])

print('Orders per category:')
for cat, order_ids in sorted(rows_by_category.items()):
    print(f'  {cat}: {len(order_ids)} orders')

---

## 4.2 Classes

A **class** groups related data and functions into one unit. You have already used classes — a `Path` is a class, a `datetime` is a class. Now you will write your own.

Why use a class in a pipeline? When you want to track state — for example, how many rows were read, how many were dropped, and why — a class is cleaner than passing multiple variables between functions.

In [ ]:
# A class that tracks what the cleaning pipeline did
class CleaningReport:
    def __init__(self, source_file):
        self.source_file = source_file
        self.rows_read          = 0
        self.duplicates_removed = 0
        self.invalid_dropped    = 0
        self.totals_filled      = 0

    def clean_rows(self):
        return self.rows_read - self.duplicates_removed - self.invalid_dropped

    def summary(self):
        lines = [
            f'Source:               {self.source_file}',
            f'Rows read:            {self.rows_read}',
            f'Duplicates removed:   {self.duplicates_removed}',
            f'Invalid dropped:      {self.invalid_dropped}',
            f'Missing totals filled: {self.totals_filled}',
            f'Clean rows output:    {self.clean_rows()}',
        ]
        return '\n'.join(lines)

# Create an instance and update it as cleaning runs
report = CleaningReport('sales_q1.csv')
report.rows_read          = 26
report.duplicates_removed = 1
report.invalid_dropped    = 1
report.totals_filled      = 2

print(report.summary())

In [ ]:
# Key point: each instance is completely independent
# Changing one report does not affect another

report_q1 = CleaningReport('sales_q1.csv')
report_q2 = CleaningReport('sales_q2.csv')

report_q1.rows_read = 26
report_q2.rows_read = 25

print(f'Q1 rows read: {report_q1.rows_read}')   # 26
print(f'Q2 rows read: {report_q2.rows_read}')   # 25  — unchanged by the Q1 update

# They share the same class definition but hold separate data
print(f'\nSame class: {type(report_q1) is type(report_q2)}')    # True
print(f'Same object: {report_q1 is report_q2}')                 # False

In [ ]:
# A class with methods that do the work
class CategoryCleaner:
    '''Normalises category names and fixes known typos.'''

    def __init__(self, typo_map=None):
        self.typo_map = typo_map or {}   # dict of wrong -> correct spellings
        self.corrections_made = 0

    def clean(self, value):
        '''Clean one category value. Returns the corrected string.'''
        normalised = str(value).strip().title()
        corrected = self.typo_map.get(normalised, normalised)
        if corrected != normalised:
            self.corrections_made += 1
        return corrected

typos = {
    'Electrnics':    'Electronics',
    'Home & Kithen': 'Home & Kitchen',
}
cleaner = CategoryCleaner(typos)

test_values = ['ELECTRONICS', 'Electrnics', '  clothing  ', 'Home & Kithen', 'clothing']
for v in test_values:
    result = cleaner.clean(v)
    print(f'  {v!r:25} -> {result}')

print(f'\nTotal corrections made: {cleaner.corrections_made}')

---

## 4.3 Putting It Together — A Structured Pipeline

Here is a complete cleaning pipeline using everything from this week: functions, classes, pathlib, csv, exception handling, and datetime.

In [ ]:
def clean_price(value):
    try:
        return float(str(value).replace('$', '').replace(',', '').strip())
    except (ValueError, TypeError):
        return None

def clean_row(row, cat_cleaner, report):
    '''Clean one row. Returns the cleaned dict, or None if the row is invalid.'''
    row['customer_name'] = row['customer_name'].strip()
    row['product']       = row['product'].strip()
    row['category']      = cat_cleaner.clean(row['category'])
    row['date']          = parse_date(row['date'])

    row['unit_price'] = clean_price(row['unit_price'])
    try:
        row['quantity'] = int(float(row['quantity']))
    except (ValueError, TypeError):
        row['quantity'] = None

    if row['total'] == '' or row['total'] is None:
        if row['quantity'] and row['unit_price']:
            row['total'] = row['quantity'] * row['unit_price']
            report.totals_filled += 1
    else:
        row['total'] = clean_price(row['total'])

    # Return None to signal the row should be dropped
    if not row['quantity'] or row['quantity'] <= 0:
        report.invalid_dropped += 1
        return None

    return row

print('Helper functions defined.')

In [ ]:
def run_pipeline(input_path, output_path, typo_map=None):
    '''Read, clean, and write a quarterly sales CSV. Returns a CleaningReport.'''
    report  = CleaningReport(input_path.name)
    cleaner = CategoryCleaner(typo_map)

    # Read
    with open(input_path, 'r', newline='', encoding='utf-8') as f:
        raw_rows = list(csv.DictReader(f))
    report.rows_read = len(raw_rows)

    # Deduplicate
    seen   = set()
    unique = []
    for row in raw_rows:
        if row['order_id'] in seen:
            report.duplicates_removed += 1
        else:
            seen.add(row['order_id'])
            unique.append(row)

    # Clean
    clean_rows = []
    for row in unique:
        cleaned = clean_row(row, cleaner, report)
        if cleaned is not None:
            clean_rows.append(cleaned)

    # Write
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(clean_rows[0].keys()))
        writer.writeheader()
        writer.writerows(clean_rows)

    return report

print('run_pipeline() defined.')

In [ ]:
# Run the pipeline on ONE file
TYPO_MAP = {
    'Electrnics':    'Electronics',
    'Home & Kithen': 'Home & Kitchen',
}

report = run_pipeline(
    input_path  = RAW / 'sales_q1.csv',
    output_path = Path('../data/clean/sales_q1_pipeline.csv'),
    typo_map    = TYPO_MAP
)
print(report.summary())

In [ ]:
# Run the pipeline on ALL THREE quarterly files
# Collect a separate CleaningReport for each file — they are independent objects

CLEAN = Path('../data/clean')
all_reports = []

for input_path in sorted(RAW.glob('*.csv')):
    output_path = CLEAN / (input_path.stem + '_pipeline.csv')
    report = run_pipeline(input_path, output_path, typo_map=TYPO_MAP)
    all_reports.append(report)
    print(f'--- {input_path.name} ---')
    print(report.summary())
    print()

# Total rows across all three files
total_clean = sum(r.clean_rows() for r in all_reports)
total_read  = sum(r.rows_read for r in all_reports)
print(f'Grand total: {total_clean} clean rows from {total_read} raw rows')

---

## 4.4 Modules — Sharing Code Across Files

When your cleaning functions are useful across multiple notebooks or scripts, move them into a `.py` file. Any file ending in `.py` is a module. You import from it with `from module_name import function_name`.

A typical Week 3 project layout:

```
week3/
├── cleaners.py           <- clean_price(), parse_date(), CategoryCleaner
├── pipeline.py           <- run_pipeline()
└── labs/
    └── lab3_clean_csv_files.ipynb  <- imports from cleaners and pipeline
```

The lab notebook would start with:
```python
from cleaners import clean_price, parse_date, CategoryCleaner
from pipeline import run_pipeline
```

This is the pattern used in professional Python projects. You will apply it in the lab.

In [ ]:
# Write a small cleaners.py module
cleaners_src = '''"""Reusable cleaning functions for the Week 3 sales pipeline."""
from datetime import datetime


def parse_date(date_str):
    """Parse a date string in multiple formats. Returns YYYY-MM-DD or None."""
    formats = ['%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y', '%b %d %Y']
    for fmt in formats:
        try:
            return datetime.strptime(str(date_str).strip(), fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return None


def clean_price(value):
    """Remove currency symbols and convert to float. Returns None on failure."""
    try:
        return float(str(value).replace('$', '').replace(',', '').strip())
    except (ValueError, TypeError):
        return None


class CategoryCleaner:
    """Normalises category names and fixes known typos."""

    def __init__(self, typo_map=None):
        self.typo_map = typo_map or {}
        self.corrections_made = 0

    def clean(self, value):
        normalised = str(value).strip().title()
        corrected = self.typo_map.get(normalised, normalised)
        if corrected != normalised:
            self.corrections_made += 1
        return corrected
'''

mod_path = Path('cleaners.py')
mod_path.write_text(cleaners_src, encoding='utf-8')
print(f'Written: {mod_path}')

In [ ]:
# Import from the module we just wrote
from cleaners import parse_date, clean_price, CategoryCleaner

print(parse_date('2023/07/14'))       # 2023-07-14
print(clean_price('$350.00'))         # 350.0

cat = CategoryCleaner({'Electrnics': 'Electronics'})
print(cat.clean('Electrnics'))        # Electronics

# Clean up
mod_path.unlink()
print('\nModule file deleted.')

---

## Key Takeaways

1. **Standard library**: `datetime` for date parsing, `Counter` for counting, `defaultdict` for grouping. All built into Python — no installation needed.
2. **`datetime.strptime(s, fmt)`** parses a date string. **`.strftime(fmt)`** formats it. Use `%B` for the full month name, `%A` for the full weekday name. Subtract two `datetime` objects to get the number of days between them.
3. **Each class instance is independent.** Creating two `CleaningReport` objects gives you two separate containers — updating one does not affect the other.
4. **Classes** bundle data (stored as attributes) and behaviour (methods). Use them when multiple functions need to share the same state — like a report that accumulates counts as the pipeline runs.
5. **`__init__`** runs when you create an instance with `MyClass(...)`. Use it to set up the initial state.
6. **Looping over files with `glob('*.csv')`** and calling `run_pipeline()` for each one scales to any number of quarterly files without changing the code — just add a new file to the folder.
7. **Modules** (`.py` files) make cleaning functions reusable across notebooks. Import with `from module import function`.
8. A structured pipeline has three clear stages: **read**, **clean**, **write**. Each stage can be a function, and the whole thing can be wrapped in a class when you need to track state across stages.